# Hot/Cold FCT Plots

Change workload-specific settings in `WORKLOAD_CONFIGS`; change experiment-wide settings in the shared constants.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from fct_parallel import run_fct_plot as run_single_fct_plot
from plot_utils import (
    DEFAULT_DPI,
    FIG_DIR,
    FONT_SIZE,
    TEXT_WIDTH,
    add_flow_size_regions,
    add_once,
    adjust_subplot_widths,
    metric_yticks,
    save_and_trim,
    style_axis,
)

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
XTICKS_HD = [10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
XTICKS_DM = [10**2, 10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
YTICKS_HD_AVG = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3]
YTICKS_HD_P99 = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4]
LONG_FIG_RATIO = 0.225
CUTOFF = 60_000_000
DEFAULT_ALPHA = 0.18
FCT_MAX_WORKERS = 3
VERBOSE = False

EXP_NAME = "FCT_skewed"
NETWORKS = ["cbb_108i", "opera_ecmp", "clos_prio"]
LABELS = ["CBB-Net", "Opera", "3:1 Fat-tree"]
VARIABLE_NAME = "theta"
VARIABLE_VALUES = ["0.10", "0.20", "0.30", "0.40", "0.50"]
SEEDS = ["1", "2", "3"]
METRICS = ["avg", "p99"]

WORKLOAD_CONFIGS = {
    "HD": {
        "file_template": "../../results/FCT_skewed/log_{network}_HD_5.00pload_{variable}theta_0.80phi_seed={seed}.txt",
        "xticks": XTICKS_HD,
        "xlim": (XTICKS_HD[0], XTICKS_HD[-1]),
        "fig_suffix": "HD",
        "marker": "D",
    },
    "DM": {
        "file_template": "../../results/FCT_skewed/log_{network}_DM_5.00pload_{variable}theta_0.80phi_seed={seed}.txt",
        "xticks": XTICKS_DM,
        "xlim": (XTICKS_DM[0], 2 * max(XTICKS_DM)),
        "fig_suffix": "DM",
        "marker": "o",
    },
}


In [ ]:
# Common reading and trimming helpers are imported from fct_parallel.py and plot_utils.py.


In [ ]:
def plot_theta_lines(ax, df, variable_name, variable_values, handles, labels, marker):
    """Plot one hot/cold curve for each theta value."""
    colormap = plt.cm.Dark2
    colors = [colormap(i) for i in range(len(variable_values))]

    for i, variable in enumerate(variable_values):
        column_name = f"{variable_name.capitalize()} {variable}"
        if column_name not in df:
            continue

        (line,) = ax.plot(
            df["Size"],
            df[column_name],
            marker=marker,
            markersize=1.5,
            linewidth=0.5,
            label=f"$\u03B8$ = {round(float(variable), 2)}",
            color=colors[i],
        )
        add_once(handles, labels, line)


def plot_fct_results(
    fct_results_df,
    variable_name,
    variable_values,
    labels,
    fct_metric,
    fig_name,
    xticks,
    xlim,
    marker="o",
):
    """Plot hot/cold FCT results for one workload/metric and save the figure."""
    num_networks = len(fct_results_df)
    if len(labels) != num_networks:
        raise ValueError("labels must have one entry for each network result")

    fig_width = TEXT_WIDTH
    fig_height = LONG_FIG_RATIO * fig_width
    first_yticks = metric_yticks(fct_metric, YTICKS_HD_AVG, YTICKS_HD_P99)
    normalized_yticks = [0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8, 2]
    legend_handles = []
    legend_labels = []

    fig, axes = plt.subplots(
        1,
        num_networks,
        figsize=(fig_width, fig_height),
        sharey=False,
        dpi=DEFAULT_DPI,
    )
    axes = [axes] if num_networks == 1 else list(axes)

    for idx, (_, df) in enumerate(fct_results_df.items()):
        ax = axes[idx]
        add_flow_size_regions(
            ax,
            xlim,
            legend_handles,
            legend_labels,
            cutoff=CUTOFF,
            alpha=DEFAULT_ALPHA,
        )
        plot_theta_lines(ax, df, variable_name, variable_values, legend_handles, legend_labels, marker)
        style_axis(ax, idx, labels[idx], fct_metric, xticks, xlim, first_yticks, normalized_yticks)

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        handlelength=1.25,
        markerscale=1.0,
        loc="center right",
        fontsize=FONT_SIZE - 2,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0, 0.92, 1])
    adjust_subplot_widths(axes)

    save_and_trim(f"{FIG_DIR}/{fig_name}.png", dpi=DEFAULT_DPI)
    plt.show()


In [ ]:
def run_fct_plot(workload, fct_metric, display_network="opera_ecmp"):
    """Compute averaged FCT data and plot one hot/cold workload/metric pair."""
    workload = workload.upper()
    if workload not in WORKLOAD_CONFIGS:
        raise ValueError(f"Unknown workload {workload}; choose from {list(WORKLOAD_CONFIGS)}")

    config = WORKLOAD_CONFIGS[workload]

    def plot_with_workload_marker(*args, **kwargs):
        return plot_fct_results(*args, marker=config["marker"], **kwargs)

    return run_single_fct_plot(
        fct_metric,
        exp_name=f"{EXP_NAME}_{config['fig_suffix']}",
        file_template=config["file_template"],
        networks=NETWORKS,
        variable_name=VARIABLE_NAME,
        variable_values=VARIABLE_VALUES,
        seeds=SEEDS,
        labels=LABELS,
        xticks=config["xticks"],
        xlim=config["xlim"],
        plot_fct_results_func=plot_with_workload_marker,
        display_func=display,
        display_network=display_network,
        max_workers=FCT_MAX_WORKERS,
        verbose=VERBOSE,
    )


def run_workload(workload):
    """Run both average and p99 plots for one workload."""
    return {metric: run_fct_plot(workload, metric) for metric in METRICS}


def run_all_workloads():
    """Run average and p99 plots for every configured hot/cold workload."""
    return {workload: run_workload(workload) for workload in WORKLOAD_CONFIGS}


## Run Selected Plots


In [ ]:
hd_results = run_workload("HD")


In [ ]:
dm_results = run_workload("DM")
